# 001 · if / elif / else

> **Syntax › Control Flow** · Python 3.14

Branching in Python looks familiar, but the syntax is leaner than C#: no parentheses, no braces, and a dedicated `elif` keyword. Conditions also don't have to be `bool` — any object can be tested for *truthiness*.

**In this notebook you will learn:**

- the `if` / `elif` / `else` syntax and how it maps to C#'s `if` / `else if` / `else`
- the conditional expression `a if cond else b` (C#'s ternary `?:`)
- how truthiness lets you write `if items:` instead of `if (items.Count > 0)`
- guard clauses vs deeply nested `if`s
- `pass` — the "do nothing" statement for empty blocks

## C# mental model

| C# | Python |
|---|---|
| `if (x > 0) { ... }` | `if x > 0:` + indented block |
| `else if (...)` | `elif ...:` |
| `else { ... }` | `else:` |
| `cond ? a : b` | `a if cond else b` |
| `{ }` (empty block) or `;` | `pass` |
| condition must be `bool` | condition can be **any** object (truthiness) |

## Basic syntax

```csharp
// C#
int temperature = 23;
if (temperature > 30)
{
    Console.WriteLine("hot");
}
else if (temperature > 15)
{
    Console.WriteLine("pleasant");
}
else
{
    Console.WriteLine("cold");
}
```

In Python, a colon ends the condition line and **indentation** defines the block. `else if` is spelled `elif`:

In [ ]:
temperature = 23

if temperature > 30:
    print("hot")
elif temperature > 15:
    print("pleasant")
else:
    print("cold")

## No parentheses needed

Parentheses around the condition are allowed but unidiomatic — linters will flag them. Use parentheses only for grouping or to wrap a long condition across several lines.

In [ ]:
age, has_ticket = 20, True

if (age >= 18):                       # works, but not Pythonic
    print("adult")

if age >= 18 and has_ticket:          # idiomatic: and / or / not instead of && / || / !
    print("may enter")

if (age >= 18
        and has_ticket
        and not age > 99):            # parentheses let a long condition span lines
    print("still may enter")

Python also supports **chained comparisons**, which read like maths and remove a common `&&` pattern:

```csharp
// C#
if (0 <= score && score <= 100) { ... }
```

In [ ]:
score = 87

if 0 <= score <= 100:                 # same as 0 <= score and score <= 100
    print("valid score")

⚠️ **Gotcha:** `elif` is a single keyword. Writing `else if` is a `SyntaxError` on one line — and if you put the `if` on the next line inside the `else` block you get nested code that works but is ugly and adds a level of indentation for every branch.

In [ ]:
x = 5

# else if x > 3:      # SyntaxError — Python has no "else if"

if x > 10:
    print("big")
else:
    if x > 3:                         # legal but needlessly nested — use elif
        print("medium")

## The conditional expression (ternary)

C#'s `cond ? a : b` becomes `a if cond else b` in Python. Note the **order**: the "true" value comes first, then the condition.

```csharp
// C#
var label = count == 1 ? "item" : "items";
```

In [ ]:
count = 3
label = "item" if count == 1 else "items"
print(f"{count} {label}")

# Chaining works, but readability drops fast — prefer if/elif for more than two outcomes
size = "small" if count < 2 else "medium" if count < 5 else "large"
print(size)

Both branches are expressions, and only the chosen one is evaluated — exactly like C#:

In [ ]:
items = []
first = items[0] if items else None   # items[0] is never evaluated when items is empty
print(first)

## Truthiness in conditions

In C# an `if` condition must be a `bool`. In Python, **any object** can be used: the following are *falsy*, and almost everything else is *truthy*:

- `False`, `None`
- zero: `0`, `0.0`, `0j`
- empty collections and strings: `""`, `[]`, `()`, `{}`, `set()`, `range(0)`

*(see Data Types › 004 · Booleans and Truthiness for the full rules)*

```csharp
// C#
if (items.Count > 0) { ... }
if (!string.IsNullOrEmpty(name)) { ... }
```

In [ ]:
cart = ["apple"]
name = ""

if cart:                              # C#: cart.Count > 0
    print("cart has items")

if not name:                          # C#: string.IsNullOrEmpty(name)
    print("name is empty")

⚠️ **Gotcha:** truthiness conflates "missing" and "empty/zero". If `0` or `""` is a valid value, test for `None` explicitly with `is None` / `is not None`.

In [ ]:
def describe_discount(discount: float | None) -> str:
    if not discount:                  # BUG: treats 0.0 the same as None
        return "no discount set"
    return f"{discount:.0%} off"

def describe_discount_fixed(discount: float | None) -> str:
    if discount is None:              # only None means "not set"
        return "no discount set"
    return f"{discount:.0%} off"

print(describe_discount(0.0))         # wrong: 0% is a real value
print(describe_discount_fixed(0.0))   # 0% off

## Nested ifs vs guard clauses

Deep nesting is harder to read in Python than in C# because every level costs four spaces of indentation. The same refactoring applies in both languages: **return early** for the invalid cases (guard clauses) and keep the "happy path" un-indented.

```csharp
// C#
string Ship(Order? order)
{
    if (order is null) return "no order";
    if (!order.IsPaid) return "awaiting payment";
    return $"shipping {order.Id}";
}
```

In [ ]:
def ship_nested(order: dict | None) -> str:
    if order is not None:
        if order["paid"]:
            if order["items"]:
                return f"shipping order {order['id']}"
            else:
                return "order is empty"
        else:
            return "awaiting payment"
    else:
        return "no order"

def ship_guarded(order: dict | None) -> str:
    if order is None:
        return "no order"
    if not order["paid"]:
        return "awaiting payment"
    if not order["items"]:
        return "order is empty"
    return f"shipping order {order['id']}"   # happy path, no nesting

order = {"id": 42, "paid": True, "items": ["book"]}
print(ship_nested(order))
print(ship_guarded(order))
print(ship_guarded({"id": 7, "paid": False, "items": []}))

## `pass` — the empty block

Python's grammar requires every block to contain at least one statement. Where C# would use `{ }`, Python uses `pass`. It's common as a placeholder while sketching code.

```csharp
// C#
if (x > 0) { }        // intentionally empty
```

In [ ]:
x = 10

if x > 0:
    pass              # TODO: handle positive numbers
else:
    print("not positive")

print("done")

`...` (the `Ellipsis` literal) is also a valid statement and is often used the same way, especially in stubs and abstract methods. A comment alone is **not** enough — the block would be empty:

In [ ]:
try:
    exec("if True:\n    # just a comment\n")
except SyntaxError as e:
    print(f"{type(e).__name__}: {e}")

if True:
    ...               # also valid

## Assigning inside a condition: the walrus operator

C# lets you assign inside a condition (`if ((n = Read()) > 0)`). Python's plain `=` can't appear in a condition, but the **walrus operator** `:=` (3.8+) can:

In [ ]:
import re

text = "order #1234 shipped"
if (m := re.search(r"#(\d+)", text)) is not None:   # assign and test in one go
    print("order number:", m.group(1))

## Summary

| Task | C# | Python |
|---|---|---|
| Branch | `if (c) { } else if (d) { } else { }` | `if c: ... elif d: ... else: ...` |
| Ternary | `c ? a : b` | `a if c else b` |
| Logical ops | `&&`, `\|\|`, `!` | `and`, `or`, `not` |
| Range check | `0 <= x && x <= 9` | `0 <= x <= 9` |
| Non-empty test | `list.Count > 0` | `if lst:` |
| Null test | `x is null` / `x is not null` | `x is None` / `x is not None` |
| Empty block | `{ }` | `pass` (or `...`) |
| Assign in condition | `if ((x = F()) > 0)` | `if (x := f()) > 0:` |

## Exercises

1. Write `grade(score)` that returns `"A"` for 90+, `"B"` for 80–89, `"C"` for 70–79 and `"F"` otherwise. Return `"invalid"` if the score is outside 0–100 (use a chained comparison).
2. Rewrite `parity = "even" if n % 2 == 0 else "odd"` as a full `if`/`else` statement, then back again.
3. Refactor this into guard clauses:
   ```python
   def withdraw(balance, amount):
       if amount > 0:
           if amount <= balance:
               return balance - amount
           else:
               return "insufficient funds"
       else:
           return "invalid amount"
   ```

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
def grade(score: int) -> str:
    if not 0 <= score <= 100:
        return "invalid"
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    else:
        return "F"

# 2
n = 7
if n % 2 == 0:
    parity = "even"
else:
    parity = "odd"
parity = "even" if n % 2 == 0 else "odd"

# 3
def withdraw(balance, amount):
    if amount <= 0:
        return "invalid amount"
    if amount > balance:
        return "insufficient funds"
    return balance - amount
```

</details>